# 自己回帰モデルとフローベースモデル

自己回帰モデルは、複数の値を一度に作る代わりに、前の値を条件として次の値を順に作るモデルです。同時分布を条件付き分布の積に分解することで、尤度を直接計算できます。

フローベースモデルは、単純な分布の点 `z` を、戻せる変換でデータ `x` へ写すモデルです。逆変換で `x` から `z` に戻し、変換が空間を広げたり縮めたりした分を log-det で補正して密度を計算します。2 つの考え方はどちらも尤度を扱えますが、生成手順と設計制約が大きく異なります。

In [ ]:
import numpy as np

rng = np.random.default_rng(25)

n = 1600
x1 = rng.normal(loc=0.8, scale=1.1, size=n)
noise = rng.normal(loc=0.0, scale=0.45, size=n)
x2 = 1.35 * x1 - 0.4 + noise
data = np.column_stack([x1, x2])

print('data shape:', data.shape)
print('mean:', np.round(data.mean(axis=0), 3))
print('std:', np.round(data.std(axis=0), 3))
print('corr:', round(np.corrcoef(data.T)[0, 1], 3))

2変数なら、連鎖律は `p(x1, x2) = p(x1) p(x2 | x1)` になります。`x1` を正規分布、`x2 | x1` を線形ガウス分布と置くと、相関を少ないパラメータで表せます。周辺分布と条件付き分布を分けることで、同時分布を一度に直接作らずに密度を計算できます。

In [ ]:
def gaussian_nll(value, mean, sigma):
    return 0.5 * np.log(2.0 * np.pi * sigma**2) + 0.5 * ((value - mean) / sigma) ** 2


def fit_linear_gaussian_ar(samples):
    x1 = samples[:, 0]
    x2 = samples[:, 1]
    mu1 = x1.mean()
    sigma1 = x1.std()
    design = np.column_stack([x1, np.ones_like(x1)])
    a, b = np.linalg.lstsq(design, x2, rcond=None)[0]
    residual = x2 - (a * x1 + b)
    sigma2 = residual.std()
    return {'mu1': mu1, 'sigma1': sigma1, 'a': a, 'b': b, 'sigma2': sigma2}


def ar_nll(params, samples):
    x1 = samples[:, 0]
    x2 = samples[:, 1]
    mean2 = params['a'] * x1 + params['b']
    loss1 = gaussian_nll(x1, params['mu1'], params['sigma1'])
    loss2 = gaussian_nll(x2, mean2, params['sigma2'])
    return float(np.mean(loss1 + loss2))

ar_params = fit_linear_gaussian_ar(data)
print({k: round(v, 3) for k, v in ar_params.items()})
print('AR NLL:', round(ar_nll(ar_params, data), 3))

自己回帰生成では、`x2` を出す前に `x1` が必要になります。次元が増えるほど、この依存関係は長い待ち列になります。尤度計算は分解しやすい一方で、生成は前の値が決まるまで次を出せません。

In [ ]:
def sample_ar(params, n, rng):
    x1 = rng.normal(params['mu1'], params['sigma1'], size=n)
    mean2 = params['a'] * x1 + params['b']
    x2 = rng.normal(mean2, params['sigma2'])
    return np.column_stack([x1, x2])

ar_samples = sample_ar(ar_params, 4000, rng)

print('sample mean:', np.round(ar_samples.mean(axis=0), 3))
print('sample std:', np.round(ar_samples.std(axis=0), 3))
print('sample corr:', round(np.corrcoef(ar_samples.T)[0, 1], 3))

フローでは、潜在変数 `z` をデータ `x` へ写す可逆変換を作ります。可逆とは、出力から入力を一意に戻せるという意味です。2次元 affine coupling は `z1` を固定し、`z2` だけを `z1` から決まる scale と shift で変換します。

`x1 = z1`

`x2 = z2 * exp(s(z1)) + t(z1)`

逆向きは `z2 = (x2 - t(x1)) * exp(-s(x1))` で計算できます。

In [ ]:
def split_theta(theta):
    return theta['slope_s'], theta['bias_s'], theta['slope_t'], theta['bias_t']


def affine_forward(z, theta):
    a_s, b_s, a_t, b_t = split_theta(theta)
    z1 = z[:, 0]
    z2 = z[:, 1]
    scale = a_s * z1 + b_s
    shift = a_t * z1 + b_t
    x1 = z1
    x2 = z2 * np.exp(scale) + shift
    x = np.column_stack([x1, x2])
    logdet = scale
    return x, logdet


def affine_inverse(x, theta):
    a_s, b_s, a_t, b_t = split_theta(theta)
    x1 = x[:, 0]
    x2 = x[:, 1]
    scale = a_s * x1 + b_s
    shift = a_t * x1 + b_t
    z1 = x1
    z2 = (x2 - shift) * np.exp(-scale)
    z = np.column_stack([z1, z2])
    logdet_inv = -scale
    return z, logdet_inv

teacher_theta = {'slope_s': 0.25, 'bias_s': -0.15, 'slope_t': 1.2, 'bias_t': -0.35}
z = rng.normal(size=(5, 2))
x, ld = affine_forward(z, teacher_theta)
z_back, ild = affine_inverse(x, teacher_theta)

print('max recovery error:', np.max(np.abs(z - z_back)))
print('max logdet sum:', np.max(np.abs(ld + ild)))

変数変換の公式により、データ `x` の対数密度は潜在変数 `z = f^{-1}(x)` の対数密度と逆変換の log-det を足して得られます。

`log p(x) = log p(z) + log |det dz/dx|`

log-det は、変換が空間を広げたり縮めたりした分の密度補正です。可逆であっても、この補正を落とすと確率密度としては正しくなりません。

In [ ]:
def standard_normal_logpdf(z):
    return -0.5 * z.shape[1] * np.log(2.0 * np.pi) - 0.5 * np.sum(z**2, axis=1)


def flow_nll(theta, samples):
    z, logdet_inv = affine_inverse(samples, theta)
    logp = standard_normal_logpdf(z) + logdet_inv
    return float(-np.mean(logp))

flow_base = rng.normal(size=(2000, 2))
flow_data, _ = affine_forward(flow_base, teacher_theta)

near_theta = {'slope_s': 0.2, 'bias_s': -0.1, 'slope_t': 1.05, 'bias_t': -0.25}
bad_theta = {'slope_s': -0.6, 'bias_s': 0.9, 'slope_t': -0.4, 'bias_t': 0.8}

for name, theta in [('teacher', teacher_theta), ('near', near_theta), ('bad', bad_theta)]:
    print(name, 'NLL:', round(flow_nll(theta, flow_data), 3))

自己回帰では条件付き分布を順に呼び出します。フローでは潜在ノイズをまとめて変換できます。小さな2次元例でも、生成手順の違いは関数呼び出しの形に現れます。

In [ ]:
def sample_flow(theta, n, rng):
    z = rng.normal(size=(n, 2))
    x, _ = affine_forward(z, theta)
    return x

flow_samples = sample_flow(teacher_theta, 4000, rng)

print('flow sample mean:', np.round(flow_samples.mean(axis=0), 3))
print('flow sample std:', np.round(flow_samples.std(axis=0), 3))
print('flow sample corr:', round(np.corrcoef(flow_samples.T)[0, 1], 3))
print('AR generation order: x1 -> x2')
print('flow generation order: z -> x in one invertible map')

高次元では、全成分を一度に動かすより、マスクで固定側と変換側を分けます。固定側から scale と shift を作れば、変換側だけを戻せるため、層全体が可逆になります。

In [ ]:
def masked_coupling_forward(x, mask, w_scale, w_shift):
    fixed = x * mask
    scale = fixed @ w_scale
    shift = fixed @ w_shift
    y = fixed + (1.0 - mask) * (x * np.exp(scale) + shift)
    logdet = np.sum((1.0 - mask) * scale, axis=1)
    return y, logdet


def masked_coupling_inverse(y, mask, w_scale, w_shift):
    fixed = y * mask
    scale = fixed @ w_scale
    shift = fixed @ w_shift
    x = fixed + (1.0 - mask) * ((y - shift) * np.exp(-scale))
    logdet_inv = -np.sum((1.0 - mask) * scale, axis=1)
    return x, logdet_inv

x4 = rng.normal(size=(6, 4))
mask = np.array([1.0, 0.0, 1.0, 0.0])
w_scale = np.array([
    [0.0, 0.15, 0.0, -0.2],
    [0.0, 0.0, 0.0, 0.0],
    [0.0, -0.1, 0.0, 0.25],
    [0.0, 0.0, 0.0, 0.0],
])
w_shift = np.array([
    [0.0, 0.4, 0.0, -0.3],
    [0.0, 0.0, 0.0, 0.0],
    [0.0, 0.2, 0.0, 0.1],
    [0.0, 0.0, 0.0, 0.0],
])

y4, ld4 = masked_coupling_forward(x4, mask, w_scale, w_shift)
x4_back, ild4 = masked_coupling_inverse(y4, mask, w_scale, w_shift)

print('masked recovery error:', np.max(np.abs(x4 - x4_back)))
print('masked logdet sum:', np.max(np.abs(ld4 + ild4)))
print('fixed dimensions unchanged:', np.allclose(x4[:, mask == 1], y4[:, mask == 1]))

自己回帰モデルは確率分布を条件付きに分解するため、複雑な依存関係を表しやすい一方、生成は順番に進みます。フローベースモデルは可逆変換として設計するため、密度計算には log-det 補正が必要になり、変換の形にも制約があります。その代わり、潜在ノイズからデータへの写像をまとめて実行しやすくなります。